# FDS Challenge 2026: Pokémon Battle Prediction

The task is to predict whether Player 1 wins a Gen 2 OU battle. For each battle we get P1's full team, P2's lead and the first 30 turns, while the battles last between 30 and 80 turns.

This notebook clones our GitHub repository, where all the code is, and goes through our work in this order:

1. Data
2. Feature engineering (two feature sets, A and B)
3. Validation with 5-fold cross-validation
4. Evaluation of the models (metrics and curves)
5. The three final submissions

| # | Submission file | Model | Private | Public | 5-fold CV |
|---|---|---|---|---|---|
| 1 | `submission_blend_lr_hgb.csv` | Logistic regression + HistGradientBoosting on feature set B | 0.68200 | 0.69333 | 0.685 |
| 2 | `submission_blend.csv` | First version of the same blend | 0.68114 | 0.69933 | 0.689 |
| 3 | `submission_merged_ensemble.csv` | The blend plus LightGBM and XGBoost on feature set A | 0.68085 | 0.69733 | 0.688 |

To run it on Kaggle, the competition data has to be attached and Internet has to be on (for `git clone` and `pip install`).

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/lauritsrossebo/fds-pokemon-battle-prediction.git'

ON_KAGGLE = Path('/kaggle/working').exists()
if ON_KAGGLE:
    REPO_DIR = Path('/kaggle/working/fds-pokemon-battle-prediction')
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
    OUT_DIR, ENV_ROOT = Path('/kaggle/working'), Path('/tmp/envs')
else:
    REPO_DIR = Path.cwd()                      # run locally from the repository folder
    OUT_DIR, ENV_ROOT = REPO_DIR / 'output', REPO_DIR / '.envs'
OUT_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(REPO_DIR))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LogisticRegression

from pokefds import evaluation as ev
from pokefds.models import (blend_scores, cross_validate_model, hist_gb_model, lightgbm_model,
                            logistic_model, random_forest_model, xgboost_model)
from pokefds.pinned import run_pinned
from pokefds.submissions import as_arrays, feature_set_a, feature_set_b, load_competition_data

## 1. Data

10,000 training battles with the result and 5,000 test battles without it. The classes are balanced, so accuracy is a reasonable metric and guessing gives 50%.

In [ ]:
data = load_competition_data()
train_data, test_data = data['train'], data['test']
print(len(train_data), 'training battles,', len(test_data), 'test battles')
print('share of P1 wins in training:', np.mean([b['player_won'] for b in train_data]))
print('turns per battle:', {len(b['battle_timeline']) for b in train_data})

## 2. Feature engineering

We built two feature sets independently and later combined them. Both use only the competition files. The species table (base stats and types) and the move vocabulary are built from `train.jsonl` only.

**Feature set B (233 features, `pokefds/features_b.py`).** We go through the 30 turns in order and keep the last known HP and status of every Pokémon on both sides. P2's Pokémon that never appear are counted as having full HP. From this we compute:
- the state at turn 30: fainted Pokémon, remaining team HP, Pokémon with low HP, status counts, sleep from Rest separated from sleep caused by the opponent, Pokémon that can still fight, remaining base stats weighted by HP
- how the battle went: switches, turns without a move, move categories, use of important Gen 2 moves (Spikes, Rest, Explosion, Rapid Spin, ...), boosts, damage dealt, move power adjusted for STAB and the type chart
- momentum: the team HP difference at turns 10, 20 and 30, its mean, min and max, and its slope
- the two active Pokémon at turn 30: HP, status, boosts, speed and type matchup
- static information: base stats and types of P1's team, P2's lead, and how common each team's species are in the training data

**Feature set A (291 features, `pokefds/features.py` and `features_ext.py`).** Similar ideas with a different implementation, plus sweeper and wall counts, a team-vs-team STAB matchup, held items, HP healed, turns asleep, boosts per stat, strength and matchups of the surviving Pokémon, and counts of the 60 most common moves.

In [ ]:
A_tr, A_te, A_cols = feature_set_a(data)
B_tr, B_te, B_cols = feature_set_b(data)
assert (A_tr['battle_id'].values == B_tr['battle_id'].values).all()

y = B_tr['player_won'].astype(int).values
XA, _ = as_arrays(A_tr, A_te, A_cols)
XB, _ = as_arrays(B_tr, B_te, B_cols)
print(f'A: {len(A_cols)} features | B: {len(B_cols)} features')

hp_rule = (B_tr['hpdiff_t30'].values > 0) == y
print(f'predicting that the player with more team HP at turn 30 wins: {hp_rule.mean():.4f} accuracy')

## 3. Validation

All comparisons are made with stratified 5-fold cross-validation on the training set, repeated with two seeds (42 and 7). We keep the out-of-fold probabilities, so every battle is predicted by a model that did not see it, and a blend can be scored on the same folds as its members. The test set and the public leaderboard were not used to choose features or models.

The four models of the final submissions:
- logistic regression on standardised features, with strong L2 regularisation (`C = 0.02`), on feature set B
- HistGradientBoosting (600 iterations, learning rate 0.03, 15 leaves, 50 samples per leaf), on feature set B
- LightGBM (800 trees, learning rate 0.02, 31 leaves, 30% of the features per tree), on feature set A
- XGBoost (700 trees, learning rate 0.02, depth 4), on feature set A

A blend is the average of the predicted probabilities of its members.

In [ ]:
SEEDS = (42, 7)
runs = {
    'Logistic regression (B)':   (logistic_model, XB),
    'HistGradientBoosting (B)':  (hist_gb_model, XB),
    'LightGBM (A)':              (lightgbm_model, XA),
    'XGBoost (A)':               (xgboost_model, XA),
}
BLEND = 'Blend LR + HGB (submissions 1 and 2)'
ENSEMBLE = 'Ensemble of the four models (submission 3)'
blends = {BLEND: list(runs)[:2], ENSEMBLE: list(runs)}

scores, oof = {}, {}
for name, (make, X) in runs.items():
    scores[name], oof[name] = cross_validate_model(make(), X, y, seeds=SEEDS)
for name, members in blends.items():
    oof[name] = np.mean([oof[m] for m in members], axis=0)
    scores[name] = blend_scores([oof[m] for m in members], y, seeds=SEEDS)

cv_table = pd.DataFrame({'CV accuracy': {k: v.mean() for k, v in scores.items()},
                         'std over folds': {k: v.std() for k, v in scores.items()}})
display(cv_table.round(4))

For comparison we also score two simpler models on the same folds: the starter notebook (logistic regression on six team statistics) and a random forest on feature set A.

In [ ]:
STARTER = 'Starter baseline (6 features)'
FOREST = 'Random forest (A)'
X0 = ev.starter_features(train_data).values
_, o = cross_validate_model(LogisticRegression(max_iter=1000), X0, y)
oof[STARTER] = o
_, o = cross_validate_model(random_forest_model(), XA, y)
oof[FOREST] = o

## 4. Evaluation

All numbers in this section come from the out-of-fold predictions of the first seed (42).

### Metrics

Accuracy is the competition metric. ROC AUC, log loss and the Brier score look at the predicted probabilities instead of only the predicted class.

In [ ]:
order = [STARTER, FOREST] + list(runs) + [BLEND, ENSEMBLE]
p = {name: oof[name][0] for name in order}        # out-of-fold probabilities, seed 42
metrics = ev.metrics_table(p, y)
display(metrics.round(4))

print('Confusion matrix of the ensemble:')
display(ev.confusion_table(p[ENSEMBLE], y))

### ROC curves

The starter baseline, the single rule "more team HP at turn 30 wins" (using the HP difference as score) and the final ensemble.

In [ ]:
roc_inputs = {'Starter baseline': p[STARTER],
              'HP difference at turn 30': B_tr['hpdiff_t30'].values,
              'Ensemble of the four models': p[ENSEMBLE]}
ev.plot_roc(roc_inputs, y, save_path=OUT_DIR / 'roc_curves.png')
plt.show()

### Calibration and confidence

The first figure checks whether the predicted probabilities can be read as probabilities: among the battles where the ensemble predicts about 70% for P1, did P1 win about 70% of them? The second figure shows the accuracy for groups of battles with similar confidence.

In [ ]:
ev.plot_calibration(p[ENSEMBLE], y, save_path=OUT_DIR / 'calibration.png')
plt.show()
ev.plot_accuracy_by_confidence(p[ENSEMBLE], y, save_path=OUT_DIR / 'accuracy_by_confidence.png')
plt.show()

### Learning curve

Accuracy of the LR + HGB blend when it is trained on a part of each training fold. The gap between the two lines shows how much the blend overfits, and the slope of the validation line shows whether more battles would still help.

In [ ]:
curve = ev.learning_curve_blend(XB, y)
display(curve.round(4))
ev.plot_learning_curve(curve, save_path=OUT_DIR / 'learning_curve.png')
plt.show()

### Which features the linear model uses

The largest standardised coefficients of the logistic regression fitted on all training battles (feature set B).

In [ ]:
lr_full = logistic_model().fit(XB, y)
coefs = pd.Series(lr_full[-1].coef_[0], index=B_cols)
ev.plot_top_coefficients(coefs, save_path=OUT_DIR / 'top_coefficients.png')
plt.show()

### Cross-validation against the leaderboard

All the submissions we made during the competition, from the simplest model to the final ones, with the CV accuracy we had measured before submitting and the public and private leaderboard scores. The rows marked "sub." are our three final submissions. We did not record a CV score for the model on the union of both feature sets.

In [ ]:
stages = pd.DataFrame(
    [('Random forest baseline',                 0.621, 0.63400, 0.60771),
     ('Gen 2 features + random forest',         0.668, 0.68066, 0.65971),
     ('Blend LR + HGB, first version (sub. 2)', 0.689, 0.69933, 0.68114),
     ('Blend LR + HGB (sub. 1)',                0.685, 0.69333, 0.68200),
     ('LightGBM + XGBoost + LR on A and B',     np.nan, 0.69200, 0.68342),
     ('Merged ensemble (sub. 3)',               0.688, 0.69733, 0.68085)],
    columns=['submission', 'CV', 'public', 'private'])
display(stages)
ev.plot_scores_by_stage(stages, save_path=OUT_DIR / 'scores_by_stage.png')
plt.show()

## 5. Final submissions

For each submission the models are refit on all 10,000 training battles and predict the test set. The predicted probabilities are averaged and thresholded at 0.5.

We computed the three files on different computers with different versions of scikit-learn, numpy and pandas, and we found that the versions change a few percent of the predictions. To regenerate each file as it was uploaded, `run_pinned` runs its recipe (`pokefds/submissions.py`) in a separate Python process with the library versions it was made with (`requirements/`). The versions are installed into a separate folder and do not change the environment of this notebook.

### Submission 1: `submission_blend_lr_hgb.csv`

Average of logistic regression and HistGradientBoosting, both on feature set B. The linear model is stable and uses the many difference features directly, and the boosted trees add interactions between them. This is the regenerated version of our first blend (submission 2) with newer library versions (scikit-learn 1.9.1). Private score 0.68200, public 0.69333.

In [ ]:
run_pinned('blend_lr_hgb', 'requirements/submissions_1_and_3.txt', OUT_DIR, ENV_ROOT / 'submissions_1_and_3')

### Submission 2: `submission_blend.csv`

The first version of the blend: the same features and models as submission 1, run with scikit-learn 1.7.2. The two files differ on about 3% of the battles because of the library versions. Private score 0.68114, public 0.69933.

In [ ]:
run_pinned('blend_first_version', 'requirements/submission_2.txt', OUT_DIR, ENV_ROOT / 'submission_2')

### Submission 3: `submission_merged_ensemble.csv`

Average of four models: logistic regression and HistGradientBoosting on feature set B, and LightGBM and XGBoost on feature set A. The idea was that models trained on two independently built feature sets make partly different errors. It had our best cross-validation score among the final submissions. Private score 0.68085, public 0.69733.

In [ ]:
run_pinned('merged_ensemble', 'requirements/submissions_1_and_3.txt', OUT_DIR, ENV_ROOT / 'submissions_1_and_3')

### Comparison with the uploaded files

The `submitted/` folder of the repository contains the files as we uploaded them to Kaggle. Here each regenerated file is compared with its uploaded version.

In [ ]:
for fname in ['submission_blend_lr_hgb.csv', 'submission_blend.csv', 'submission_merged_ensemble.csv']:
    sub = pd.read_csv(OUT_DIR / fname).sort_values('battle_id').reset_index(drop=True)
    uploaded = REPO_DIR / 'submitted' / fname
    if uploaded.exists():
        ref = pd.read_csv(uploaded).sort_values('battle_id').reset_index(drop=True)
        assert (sub['battle_id'] == ref['battle_id']).all()
        same = (sub['player_won'] == ref['player_won']).mean()
        print(f'{fname:32s} {len(sub)} rows, {same:.2%} of the predictions equal to the uploaded file')
    else:
        print(f'{fname:32s} {len(sub)} rows (no uploaded copy in submitted/)')